# 🎙️ Chatterbox TTS Fine-Tuning & LoRA Inference Kit

This Colab notebook provides a complete environment for fine-tuning Chatterbox TTS models with your custom dataset using either **LoRA** (efficient, recommended for <10h data) or **Full Fine-Tuning**.

## Features:
- ✅ GPU-accelerated training with CUDA
- ✅ LoRA support for efficient training (~60% less VRAM)
- ✅ Turbo and Standard model support
- ✅ Automatic preprocessing and model preparation
- ✅ Built-in inference and model merging

---

## Step 1: Clone Repository & Install Dependencies

In [ ]:
#@title 📦 Clone Repository and Install Dependencies
#@markdown Run this cell to clone the repository and install all required packages.

import os

# Clone the repository if not already present
if not os.path.exists("chatterbox-finetuning"):
    !git clone https://github.com/amazeble/chatterbox-finetuning.git
    %cd chatterbox-finetuning
else:
    %cd chatterbox-finetuning

# Install FFmpeg
!apt-get update -qq
!apt-get install -y ffmpeg

# Install Python dependencies
!pip install -q -r requirements.txt

print("✅ Dependencies installed successfully!")

## Step 2: Configure Training Settings

In [ ]:
#@title ⚙️ Configuration Settings
#@markdown Adjust these settings based on your needs before running setup.

#@markdown ### Model Selection
is_turbo = True  #@param {type:"boolean"}
#@markdown - **Turbo Mode** (`True`): GPT-2 based, faster training, BPE tokenizer
#@markdown - **Standard Mode** (`False`): Llama-based, grapheme tokenizer

#@markdown ### Training Strategy
is_lora = True  #@param {type:"boolean"}
#@markdown - **LoRA** (`True`): Recommended for datasets <10 hours, uses ~60% less VRAM
#@markdown - **Full Fine-Tune** (`False`): For massive datasets >10 hours

#@markdown ### Dataset Settings
project_name = "Elise"  #@param {type:"string"}
#@markdown Your project name (used for organizing files)

ljspeech_format = True  #@param {type:"boolean"}
#@markdown Set to `True` if your dataset is in LJSpeech format (metadata.csv)

# Update config file with settings
config_content = f"""
# Auto-generated configuration from Colab
is_turbo: bool = {str(is_turbo)}
is_lora: bool = {str(is_lora)}
project_name: str = \"{project_name}\"
ljspeech = {str(ljspeech_format)}
preprocess: Optional[Literal[True, False, \"auto\"]] = True
"""

print(f"Configuration set:")
print(f"  - Turbo Mode: {is_turbo}")
print(f"  - LoRA Training: {is_lora}")
print(f"  - Project Name: {project_name}")
print(f"  - Dataset Format: {'LJSpeech' if ljspeech_format else 'File-Based'}")

## Step 3: Download and Prepare Base Models

In [ ]:
#@title 📥 Download Base Models and Prepare Tokenizer
#@markdown This downloads the necessary base models and prepares the tokenizer. **Must run before training!**

# First, update config.py with our settings
with open('src/config.py', 'r') as f:
    config_content = f.read()

# Update is_turbo setting
config_content = config_content.replace(
    'is_turbo: bool = True',
    f'is_turbo: bool = {str(is_turbo).lower()}'
)
config_content = config_content.replace(
    'is_lora: bool = True',
    f'is_lora: bool = {str(is_lora).lower()}'
)
config_content = config_content.replace(
    'project_name: str = "Elise"',
    f'project_name: str = "{project_name}"'
)
config_content = config_content.replace(
    'ljspeech = True',
    f'ljspeech = {str(ljspeech_format).lower()}'
)

with open('src/config.py', 'w') as f:
    f.write(config_content)

print("✅ Configuration updated in src/config.py")

# Clean pretrained_models if switching modes
if os.path.exists("pretrained_models"):
    import shutil
    shutil.rmtree("pretrained_models")
    print("🗑️ Cleaned old pretrained_models directory")

# Run setup.py
print("\n⏳ Downloading and preparing models...\n")
!python setup.py

print("\n✅ Setup complete! Check the output above for the new_vocab_size value.")

## Step 4: Upload Your Dataset

In [ ]:
#@title 📁 Upload Your Dataset
#@markdown Choose one of the following methods to upload your dataset:

#@markdown ### Option A: Upload Files Directly (for small datasets)
from google.colab import files
import os
import zipfile

# Create dataset directory structure
dataset_dir = f"MyTTSDataset/{project_name}"
os.makedirs(dataset_dir, exist_ok=True)
os.makedirs(f"{dataset_dir}/wavs", exist_ok=True)

print(f"Dataset directory created: {dataset_dir}")
print("\nYou can now:")
print("1. Upload files manually using the file uploader on the left")
print("2. Or use the cell below to upload a ZIP file")

In [ ]:
#@title 📤 Upload Dataset as ZIP File
#@markdown Upload a ZIP file containing your dataset in LJSpeech format (metadata.csv + wavs/ folder)

from google.colab import files
import zipfile
import shutil

# Upload ZIP file
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"Extracting {filename}...")
        
        # Extract to dataset directory
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(f"MyTTSDataset/{project_name}")
        
        # Clean up uploaded file
        os.remove(filename)
        print(f"✅ Dataset extracted to MyTTSDataset/{project_name}/")
        
        # Verify structure
        print("\nDataset structure:")
        !ls -la "MyTTSDataset/{project_name}/"
        
        if os.path.exists(f"MyTTSDataset/{project_name}/metadata.csv"):
            print("\n✅ metadata.csv found!")
            print("First few lines:")
            !head -5 "MyTTSDataset/{project_name}/metadata.csv"
        else:
            print("\n⚠️ metadata.csv not found. Please ensure your ZIP contains it.")
    else:
        print(f"Skipping {filename} (not a ZIP file)")

## Step 5: Verify Dataset and Check GPU

In [ ]:
#@title 🔍 Verify Dataset and GPU Status

import torch
import os
import glob

# Check GPU
print("=" * 50)
print("GPU STATUS")
print("=" * 50)
if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("❌ No GPU detected! Go to Runtime > Change runtime type > GPU")

# Check dataset
print("\n" + "=" * 50)
print("DATASET STATUS")
print("=" * 50)

dataset_path = f"MyTTSDataset/{project_name}"
if os.path.exists(dataset_path):
    # Count WAV files
    wav_files = glob.glob(f"{dataset_path}/wavs/*.wav")
    print(f"📊 WAV files found: {len(wav_files)}")
    
    # Check metadata.csv
    metadata_path = f"{dataset_path}/metadata.csv"
    if os.path.exists(metadata_path):
        with open(metadata_path, 'r') as f:
            lines = f.readlines()
        print(f"📝 Metadata entries: {len(lines)}")
        print("\nSample entries:")
        for i, line in enumerate(lines[:3]):
            parts = line.strip().split('|')
            if len(parts) >= 2:
                print(f"  {i+1}. {parts[0]}: {parts[1][:50]}...")
    else:
        print("❌ metadata.csv not found!")
else:
    print(f"❌ Dataset directory not found: {dataset_path}")

# Estimate total duration
if wav_files:
    import librosa
    total_duration = 0
    for wav_file in wav_files[:10]:  # Sample first 10 files
        y, sr = librosa.load(wav_file, sr=None)
        total_duration += len(y) / sr
    avg_duration = total_duration / min(10, len(wav_files))
    estimated_total = avg_duration * len(wav_files)
    print(f"\n⏱️ Estimated total duration: ~{estimated_total/60:.1f} minutes")
    
    if estimated_total < 600:  # 10 minutes
        print("💡 Recommendation: Use LoRA training (already enabled)")
    elif estimated_total > 3600:  # 1 hour
        print("💡 You have sufficient data for full fine-tuning if needed")

## Step 6: Start Training

In [ ]:
#@title 🏋️ Start Training
#@markdown This will preprocess your dataset and start training. The process includes:
#@markdown 1. Preprocessing (audio feature extraction)
#@markdown 2. Model initialization with extended vocabulary
#@markdown 3. Training with periodic inference checks
#@markdown 
#@markdown **Expected time:** 30 mins - 2 hours depending on dataset size and GPU

print("Starting training...")
print(f"Mode: {'Turbo' if is_turbo else 'Standard'} + {'LoRA' if is_lora else 'Full Fine-Tune'}")
print(f"Project: {project_name}")
print("\nMonitoring training progress...\n")

# Run training
!python train.py

print("\n✅ Training complete!")
print("\nOutput location:")
if is_lora:
    print(f"  LoRA adapter: chatterbox_output/{project_name}/new_lang_adapter/")
else:
    model_name = "t3_turbo_finetuned.safetensors" if is_turbo else "t3_finetuned.safetensors"
    print(f"  Full model: chatterbox_output/{project_name}/{model_name}")

## Step 7: Test Inference

In [ ]:
#@title 🗣️ Test Speech Synthesis (Inference)
#@markdown Generate speech using your trained model. Make sure you have a reference audio file.

import os
from IPython.display import Audio, display

# Check for speaker reference
ref_dir = "speaker_reference"
if not os.path.exists(ref_dir):
    os.makedirs(ref_dir)
    print(f"Created {ref_dir}/ directory")
    print("\n⚠️ Please upload a reference audio file (3-10 seconds clean speech)")
    print(f"   Upload it to: {ref_dir}/reference.wav")
else:
    ref_files = [f for f in os.listdir(ref_dir) if f.endswith('.wav')]
    if ref_files:
        print(f"Found reference files: {ref_files}")
        
        # Update inference.py with test text
        test_text = "Hello, this is a test of my custom voice model."  #@param {type:"string"}
        ref_file = ref_files[0]  #@param {type:"string"}
        
        # Modify inference.py temporarily
        with open('inference.py', 'r') as f:
            inf_content = f.read()
        
        # Replace test text
        inf_content = inf_content.replace(
            'TEXT_TO_SAY = "Merhaba, sesimi geliştirmem oldukça uzun zaman aldı ve şimdi sahip olduğuma göre, sessiz kalmayacağım."',
            f'TEXT_TO_SAY = "{test_text}"'
        )
        
        with open('inference.py', 'w') as f:
            f.write(inf_content)
        
        print(f"\nRunning inference with:")
        print(f"  Text: {test_text}")
        print(f"  Reference: {ref_dir}/{ref_file}")
        print("\nGenerating speech...\n")
        
        # Run inference
        !python inference.py
        
        # Play output
        if os.path.exists("output.wav"):
            print("\n✅ Generated output.wav")
            display(Audio("output.wav"))
        else:
            print("\n❌ Output file not generated. Check error messages above.")
    else:
        print(f"\n⚠️ No .wav files found in {ref_dir}/")
        print("Please upload a reference audio file (3-10 seconds of clean speech)")

## Step 8: Merge LoRA Adapter (Optional)

In [ ]:
#@title 📦 Merge LoRA Adapter into Base Model
#@markdown If you used LoRA training and are satisfied with the results, merge the adapter into a standalone model file.

if is_lora:
    print("Merging LoRA adapter into base model...")
    print("This creates a single .safetensors file ready for deployment.\n")
    
    !python merge_lora.py
    
    print("\n✅ Merge complete!")
    merged_model = f"chatterbox_output/{project_name}/t3_turbo_finetuned_merged.safetensors" if is_turbo else f"chatterbox_output/{project_name}/t3_finetuned_merged.safetensors"
    print(f"Merged model saved to: {merged_model}")
    
    if os.path.exists(merged_model):
        size_mb = os.path.getsize(merged_model) / (1024 * 1024)
        print(f"File size: {size_mb:.1f} MB")
else:
    print("ℹ️ Skipping merge - you used full fine-tuning, not LoRA")

## Step 9: Download Trained Model

In [ ]:
#@title 💾 Download Trained Model
#@markdown Download your trained model to Google Drive or local storage.

from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# Determine what to save
output_dir = f"chatterbox_output/{project_name}"

if os.path.exists(output_dir):
    # Create backup directory in Drive
    drive_backup = f"/content/drive/MyDrive/chatterbox_models/{project_name}"
    os.makedirs(drive_backup, exist_ok=True)
    
    # Copy all output files
    for item in os.listdir(output_dir):
        src = os.path.join(output_dir, item)
        dst = os.path.join(drive_backup, item)
        
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"✅ Copied: {item} ({os.path.getsize(src) / (1024*1024):.1f} MB)")
        elif os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"✅ Copied directory: {item}/")
    
    print(f"\n📁 All files saved to: {drive_backup}")
    print("\nYou can also download individual files using the file browser on the left.")
else:
    print(f"❌ Output directory not found: {output_dir}")
    print("Make sure training has completed successfully.")

---
## Troubleshooting Tips

### Common Issues:

1. **CUDA Out of Memory**
   - Enable LoRA training (`is_lora = True`)
   - Reduce `batch_size` in `src/config.py`
   - Use a smaller dataset

2. **Poor Quality Output**
   - Ensure reference audio is clean (3-10 seconds)
   - Check dataset quality (minimal background noise)
   - Train for more epochs if dataset is small
   - For Turbo model: switch to LoRA if experiencing instability

3. **Vocabulary Mismatch Error**
   - Make sure `new_vocab_size` matches the tokenizer
   - Re-run `setup.py` after changing modes

4. **Missing Reference Audio**
   - Upload a clean 3-10 second WAV file to `speaker_reference/`
   - Use the same speaker as your training dataset for best results

---
**Need Help?** Check the README.md file or open an issue on GitHub.